In [1]:
import json
import pandas as pd

In [2]:
labels="/m/04rlf,/m/06_fw,/m/09x0r"

In [3]:
lists=labels.split(',')

In [4]:
len(lists)

3

In [5]:
len(labels.split(','))

3

In [6]:
def count_labels(labels: str) -> int:
    """
    Given a string of unparsed labels, return the number of distinct labels.

    For example:
    "/m/04rlf,/m/06_fw,/m/09x0r" -> 3
    """
    output=len(labels.split(','))
    
    return output

In [7]:
def convert_id(ID: str) -> str:
    """
    Create a function that takes in a label ID (e.g. "/m/09x0r") and returns the corresponding label name (e.g. "Speech")

    To do so, make use of the `json` library and the `data/ontology.json` file, a description of the file can be found
    at https://github.com/audioset/ontology

    While reading the file each time and looping through the elements to find a match works well enough for our
    purposes, think of ways this process could be sped up if say this function needed to be run 100000 times.
    """
    with open("data/ontology.json", "r") as f:
        ontology = json.load(f)
    ontology_dict={}
    for i in ontology:
        if "id" in i and "name" in i:
            ontology_dict[i["id"]] = i["name"]

    output=str(ontology_dict[ID])
    # TODO
    return output

In [8]:
def convert_ids(labels: str) -> str:
    """
    Using get_label_name() create a function that takes the label columns (i.e a string of comma-separated label IDs)
    and returns a string of label names, separated by pipes "|".

    For example:
    "/m/04rlf,/m/06_fw,/m/09x0r" -> "Music|Skateboard|Speech"
    """
    # TODO
    parts=[]
    for i in labels.split(','):
        name=convert_id(i)
        parts.append(name)  
    output = "|".join(parts)
    
    return output
        


In [9]:
convert_ids("/m/04rlf,/m/06_fw,/m/09x0r")

'Music|Skateboard|Speech'

In [10]:
ontology_dict["/m/04rlf"]

NameError: name 'ontology_dict' is not defined

In [11]:
convert_ids('/m/068hy,/m/07q6cd_,/m/0bt9lr,/m/0jbk')

'Domestic animals, pets|Squeak|Dog|Animal'

In [35]:
def contains_label(labels: pd.Series, label: str) -> pd.Series:
    """
    Create a function that takes a Series of strings where each string is formatted as above 
    (i.e. "|" separated label names like "Music|Skateboard|Speech") and returns a Series with just
    the values that include `label`.

    For example, given the label "Music" and the following Series:
    "Music|Skateboard|Speech"
    "Voice|Speech"
    "Music|Piano"

    the function should just return
    "Music|Skateboard|Speech"
    "Music|Piano"
    """
    # TODO
    
    valid_idx = []

    for i in range(len(labels)):
        val = labels.iloc[i]                 
        if pd.isna(val):
            continue                         
        tokens = [t.strip() for t in str(val).split("|")] 
        if label in tokens:
            valid_idx.append(labels.index[i])  

    output = labels.loc[valid_idx]       
    return output


In [36]:
def get_correlation(labels: pd.Series, label_1: str, label_2: str) -> float:
    """
    Create a function that, given a Series as described above, returns the proportion of rows
    with label_1 that also have label_2. Make use of the function you created above.

    For example, suppose the Series has 1000 values, of which 120 have label_1. If 30 of the 120
    have label_2, your function should return 0.25.
    """
    s_1=contains_label(labels, label_1)
    count_1=len(s_1)
    if count_1 == 0:
        return 0.0
    s_2=contains_label(s_1, label_2)
    count_2=len(s_2)

    output= count_2/count_1
    
    return output

In [37]:
print(count_labels("/m/04rlf,/m/06_fw,/m/09x0r"))
print(convert_id("/m/04rlf"))
print(convert_ids("/m/04rlf,/m/06_fw,/m/09x0r"))

series = pd.Series([
        "Music|Skateboard|Speech",
        "Voice|Speech",
        "Music|Piano"
    ])
print(contains_label(series, "Music"))
print(get_correlation(series, "Music", "Piano"))

3
Music
Music|Skateboard|Speech
0    Music|Skateboard|Speech
2                Music|Piano
dtype: object
0.5


In [73]:
import youtube_dl
import ffmpeg
import pandas as pd
import numpy as np
import csv
import threading
from tqdm import tqdm
from os.path import exists

In [168]:
def download_audio(YTID: str, path: str) -> None:
    """
    Create a function that downloads the audio of the Youtube Video with a given ID
    and saves it in the folder given by path. Download it as an mp3. If there is a problem downloading the file, handle the exception. 
    If a file at `path` exists, the function should return without attempting to download it again.

    ** Use the library youtube_dl: https://github.com/ytdl-org/youtube-dl/ **
    Args:
      YTID: Contains the youtube ID, the corresponding youtube video can be found at
      'https://www.youtube.com/watch?v='+YTID
      path: The path to the file where the audio will be saved
    """
    # TODO
    if os.path.exists(path):
        return None
    
    url='https://www.youtube.com/watch?v='+YTID
    
    ydl_opts = {
        "format": "bestaudio/best",
        "ffmpeg_location": "/opt/homebrew/bin",
        "outtmpl": path,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        try:
            print(f'Downloading {url}')
            ydl.download([url])
        except Exception as e:
            print(f"[WARN] {url}: {e}")

In [132]:
download_audio('6hzrDeceEKc',"6hzrDeceEKc")

https://www.youtube.com/watch?v=6hzrDeceEKc
[youtube] Extracting URL: https://www.youtube.com/watch?v=6hzrDeceEKc
[youtube] 6hzrDeceEKc: Downloading webpage
[youtube] 6hzrDeceEKc: Downloading tv simply player API JSON
[youtube] 6hzrDeceEKc: Downloading tv client config
[youtube] 6hzrDeceEKc: Downloading tv player API JSON
[info] 6hzrDeceEKc: Downloading 1 format(s): 251
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: 6hzrDeceEKc
[download] 100% of    3.54MiB in 00:00:01 at 2.50MiB/s   
[ExtractAudio] Destination: 6hzrDeceEKc.mp3
Deleting original file 6hzrDeceEKc (pass -k to keep)


In [175]:
def cut_audio(in_path: str, out_path: str, start: float, end: float) -> None:
    """
    Create a function that cuts the audio from in_path to only include the segment from start to end and saves it to out_path.

    ** Use the ffmpeg library: https://github.com/kkroening/ffmpeg-python
    Args:
      in_path: Path of the audio file to cut
      out_path: Path of file to save the cut audio

      
      start: Indicates the start of the sequence (in seconds)
      end: Indicates the end of the sequence (in seconds)  
    """
    
    file = ffmpeg.input(in_path)
    trimmed = file.filter('atrim', start=start, end=end)
    output = ffmpeg.output(trimmed, out_path)
    output.run()
    print(f"Cut successful: {out_path}")
    pass

In [133]:
cut_audio('6hzrDeceEKc.mp3', 'trim_6hzrDeceEKc.mp3', 10,20)

ffmpeg version 8.0 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

In [23]:
import re
import os
import pandas as pd
from tqdm import tqdm
from q2 import download_audio, cut_audio
from typing import List


def filter_df(csv_path: str, label: str) -> List[str]:
    """
    Write a function that takes the path to the processed csv from q1 and returns a df of only the rows 
    that contain the human readable label passed as argument

    For example:
    get_ids("audio_segments_clean.csv", "Speech")
    """
    # TODO

    df= pd.read_csv(csv_path)
    matches = contains_label(df["label_names"], label)
    mask = df["label_names"].isin(matches)
    
    return df[mask]


In [178]:
def data_pipeline(csv_path: str, label: str) -> None:
    """
    Using your previously created functions, write a function that takes a processed csv and for each video with the given label:
    (don't forget to create the audio/ folder and the associated label folder!). 
    
    It is recommended to iterate over the rows of
    filter_df()
    
    1. Downloads it to <label>_raw/<ID>.mp3
    2. Cuts it to the appropriate segment
    3. Stores it in <label>_cut/<ID>.mp3

    Use tqdm to track the progress of the download process (https://tqdm.github.io/)
    """
    df=filter_df(csv_path, label)


    raw_dir = os.path.join(f"{label}_raw")
    cut_dir = os.path.join(f"{label}_cut")

    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(cut_dir, exist_ok=True)


    for i in tqdm(range(len(df)), total=len(df), desc=f"{label}"):
        row = df.iloc[i]
        ytid= row["# YTID"]
        d_path   = os.path.join(raw_dir, f"{ytid}")
        out_path = os.path.join(cut_dir, f"{ytid}.mp3")
        start=float(row[" start_seconds"])
        end=float(row[" end_seconds"])

        try:
            download_audio(ytid, d_path)
            cut_audio(d_path+'.mp3', out_path, start, end)
        except Exception as e:
            print(f"[WARN] {ytid}: {e}")
            continue
    pass

In [179]:
data_pipeline("audio_segments_clean.csv", "Speech")

Speech:   0%|                                          | 0/5327 [00:00<?, ?it/s]

[youtube] Extracting URL: https://www.youtube.com/watch?v=--i-y1v8Hy8
[youtube] --i-y1v8Hy8: Downloading webpage
[youtube] --i-y1v8Hy8: Downloading tv simply player API JSON
[youtube] --i-y1v8Hy8: Downloading tv client config
[youtube] --i-y1v8Hy8: Downloading tv player API JSON
[info] --i-y1v8Hy8: Downloading 1 format(s): 251
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: Speech_raw/--i-y1v8Hy8
[download] 100% of  113.94KiB in 00:00:00 at 288.69KiB/s 
[ExtractAudio] Destination: Speech_raw/--i-y1v8Hy8.mp3
Deleting original file Speech_raw/--i-y1v8Hy8 (pass -k to keep)


ffmpeg version 8.0 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

Cut successful: Speech_cut/--i-y1v8Hy8.mp3
[youtube] Extracting URL: https://www.youtube.com/watch?v=-0CamVQdP_Y
[youtube] -0CamVQdP_Y: Downloading webpage
[youtube] -0CamVQdP_Y: Downloading tv simply player API JSON
[youtube] -0CamVQdP_Y: Downloading tv client config
[youtube] -0CamVQdP_Y: Downloading tv player API JSON
[info] -0CamVQdP_Y: Downloading 1 format(s): 251
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: Speech_raw/-0CamVQdP_Y
[download] 100% of   96.33KiB in 00:00:00 at 313.32KiB/s 
[ExtractAudio] Destination: Speech_raw/-0CamVQdP_Y.mp3
Deleting original file Speech_raw/-0CamVQdP_Y (pass -k to keep)


ffmpeg version 8.0 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex --e

...

[WARN] https://www.youtube.com/watch?v=2A7xNU3TtbA: ERROR: [youtube] 2A7xNU3TtbA: Video unavailable. This video is restricted. Please check the Google Workspace administrator and/or the network administrator restrictions.
[WARN] 2A7xNU3TtbA: ffmpeg error (see stderr output for detail)
[youtube] Extracting URL: https://www.youtube.com/watch?v=2B-xGbzOjYY
[youtube] 2B-xGbzOjYY: Downloading webpage
[youtube] 2B-xGbzOjYY: Downloading tv simply player API JSON
[youtube] 2B-xGbzOjYY: Downloading tv client config
[youtube] 2B-xGbzOjYY: Downloading tv player API JSON
[info] 2B-xGbzOjYY: Downloading 1 format(s): 140
[download] Sleeping 5.00 seconds as required by the site...


ERROR: unable to download video data: HTTP Error 403: Forbidden
ffmpeg version 8.0 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --ena

[WARN] https://www.youtube.com/watch?v=2B-xGbzOjYY: ERROR: unable to download video data: HTTP Error 403: Forbidden
[WARN] 2B-xGbzOjYY: ffmpeg error (see stderr output for detail)
[youtube] Extracting URL: https://www.youtube.com/watch?v=2BNSDDwGxM8
[youtube] 2BNSDDwGxM8: Downloading webpage
[youtube] 2BNSDDwGxM8: Downloading tv simply player API JSON
[youtube] 2BNSDDwGxM8: Downloading tv client config
[youtube] 2BNSDDwGxM8: Downloading tv player API JSON
[info] 2BNSDDwGxM8: Downloading 1 format(s): 251
[download] Sleeping 5.00 seconds as required by the site...
[download] Destination: Speech_raw/2BNSDDwGxM8
[download] 100% of  865.07KiB in 00:00:00 at 1.09MiB/s   
[ExtractAudio] Destination: Speech_raw/2BNSDDwGxM8.mp3


Speech:   6%|█▊                            | 323/5327 [29:45<7:40:59,  5.53s/it]


KeyboardInterrupt: 

In [25]:
filter_df("audio_segments_clean.csv", "Speech")

pandas.core.frame.DataFrame

In [188]:
def rename_files(path_cut: str, csv_path: str) -> None:
    """
    Suppose we now want to rename the files we've downloaded in `path_cut` to include the start and end times as well as length of the segment. While
    this could have been done in the data_pipeline() function, suppose we forgot and don't want to download everything again.

    Write a function that, using regex (i.e. the `re` library), renames the existing files from "<ID>.mp3" -> "<ID>_<start_seconds_int>_<end_seconds_int>_<length_int>.mp3"
    in path_cut. `csv_path` is the path to the processed csv from q1. `path_cut` is a path to the folder with the cut audio.

    For example
    "--BfvyPmVMo.mp3" -> "--BfvyPmVMo_20_30_10.mp3"

    ## BE WARY: Assume that the YTID can contain special characters such as '.' or even '.mp3' ##
    """
    
    df= pd.read_csv(csv_path)
    folder = path_cut
    files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
    for i in files:
        old_name=i
        m = re.match(r"^(.*)\.mp3$", i)
        ytid_file = m.group(1)
            
        for i in range(len(df)):
            row = df.iloc[i]
            ytid_df=row["# YTID"]
            if ytid_df==ytid_file:
                start=int(row[" start_seconds"])
                end=int(row[" end_seconds"])
                dif=int(end-start)
                new_name=f'{ytid_file}_{str(start)}_{str(end)}_{str(dif)}.mp3'
                os.rename(folder+"/"+old_name, folder+"/"+new_name)


    # TODO
    pass

In [189]:
rename_files("Speech_cut", "audio_segments_clean.csv")

In [190]:
len(filter_df("audio_segments_clean.csv", "Laughter"))

119